# Basic Language Model in Python

This notebook demonstrates how to create a basic language model using Python. The model processes a corpus of sentences and computes a language model, which is essentially a table showing the probability that a word follows a given prefix in the corpus.

## 1. Define the Corpus

In [ ]:
corpus = [
    'the cat sat on the mat',
    'the cat sat on the chair',
    'the cat ate the fish',
    'the dog sat on the mat',
    'the dog ate the bone',
    'the dog chased the cat',
]

## 2. Tokenize the Corpus

In [ ]:
# Split each sentence in the corpus into a list of words
tokenized_corpus = [sentence.split() for sentence in corpus]

## 3. Create Prefix-Word Table

In [ ]:
# This dictionary will map each prefix to a dictionary of word counts
prefix_word_table = {}

# Go through each sentence in the tokenized corpus
for sentence in tokenized_corpus:
    # Go through each position in the sentence (except the last word)
    for i in range(len(sentence) - 1):
        # Build the prefix by joining all words from the start up to position i
        prefix = ' '.join(sentence[:i + 1])
        # The next word is the word right after the prefix
        word = sentence[i + 1]
        # If we haven't seen this prefix before, create an empty dictionary for it
        if prefix not in prefix_word_table:
            prefix_word_table[prefix] = {}
        # If we haven't seen this word after this prefix before, set its count to 0
        if word not in prefix_word_table[prefix]:
            prefix_word_table[prefix][word] = 0
        # Add 1 to the count of this word following this prefix
        prefix_word_table[prefix][word] += 1

## 4. Normalize Prefix-Word Table

In [ ]:
# Go through each prefix and its word counts
for prefix, word_counts in prefix_word_table.items():
    # Calculate the total number of times any word followed this prefix
    total_count = sum(word_counts.values())
    # Divide each word's count by the total to get a probability
    for word, count in word_counts.items():
        prefix_word_table[prefix][word] = count / total_count

## 5. Visualize the Prefix-Word Table

In [ ]:
from IPython.display import display, HTML

all_words = sorted(set(w for probs in prefix_word_table.values() for w in probs))

header = '<tr><th style="border:1px solid #ccc;padding:8px;background:#2c3e50;color:white;text-align:left;">Prefix</th>'
for w in all_words:
    header += f'<th style="border:1px solid #ccc;padding:8px;background:#2c3e50;color:white;text-align:center;">{w}</th>'
header += '</tr>'

rows = ''
for prefix, word_probs in prefix_word_table.items():
    row = f'<tr><td style="border:1px solid #ccc;padding:8px;background:#f8f9fa;font-weight:bold;font-family:monospace;">{prefix}</td>'
    for w in all_words:
        prob = word_probs.get(w, 0)
        if prob > 0:
            bg = f'rgba(78,121,167,{prob})'
            row += f'<td style="border:1px solid #ccc;padding:8px;text-align:center;background:{bg};color:#222;font-weight:bold;">{prob:.0%}</td>'
        else:
            row += '<td style="border:1px solid #ccc;padding:8px;text-align:center;color:#ddd;">-</td>'
    row += '</tr>'
    rows += row

display(HTML(f'<table style="border-collapse:collapse;font-family:sans-serif;font-size:13px;">{header}{rows}</table>'))

## 6. Generate Sentence from Prefix

In [ ]:
# Import the random module (used to sample the next word based on probabilities)
import random

def generate_sentence(prefix):
    # Start the sentence with the given prefix, split into a list of words
    sentence = prefix.split()
    # Keep generating words until we can't continue
    while True:
        # Build the current prefix by joining all words generated so far
        current_prefix = ' '.join(sentence)
        # If the current prefix is not in the table, stop generating
        if current_prefix not in prefix_word_table:
            break
        # Get the probability distribution for words that can follow this prefix
        next_word_probs = prefix_word_table[current_prefix]
        # Extract the list of possible next words
        words = list(next_word_probs.keys())
        # Extract the corresponding probabilities
        probs = list(next_word_probs.values())
        # Pick a random number between 0 and 1
        r = random.random()
        # Walk through the probabilities to find which word was "selected"
        cumulative = 0
        for word, p in zip(words, probs):
            cumulative += p
            if r <= cumulative:
                next_word = word
                break
        # Add the chosen word to the sentence
        sentence.append(next_word)
    # Join all words into a single string and return the sentence
    return ' '.join(sentence)

# Generate a sentence starting with "the"
print(generate_sentence('the'))